# 🗑️ Waste Detection Training — FINAL VERSION

**Fixes from previous runs:**
- ✅ Epochs reduced to **70** — model fully converges by epoch 67 (proven from run 1)
- ✅ Saves directly to **Google Drive** every 10 epochs — session death = no loss
- ✅ Expected time: **~40 minutes** on T4 — fits in one session
- ✅ `best.pt` downloadable from Drive even if session dies

---
**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run Cell 1 (GPU check)
3. Run Cell 2 (mount Drive — click Allow)
4. Run Cell 3 (paste API key)
5. **▶ Run all** — come back in 40 min
6. `best.pt` → `My Drive/waste_model/runs/waste_v2/weights/best.pt`
---

In [ ]:
# ── Cell 1 ── GPU check + install ──────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise SystemExit('❌ No GPU! Go to: Runtime → Change runtime type → T4 GPU')
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install -q roboflow ultralytics
import ultralytics; print(f'\n✅ Ultralytics {ultralytics.__version__} ready')

In [ ]:
# ── Cell 2 ── Mount Google Drive (weights saved here LIVE during training) ──
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_RUNS = '/content/drive/MyDrive/waste_model/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print(f'✅ Drive mounted. Live checkpoints → {DRIVE_RUNS}')

In [ ]:
# ── Cell 3 ── API key ───────────────────────────────────────────────────────
# Get free key: https://app.roboflow.com → profile → API Keys
ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"  # <── only edit this line
if ROBOFLOW_API_KEY == "PASTE_YOUR_KEY_HERE":
    raise ValueError('❌ Paste your Roboflow API key!')
print('✅ Key set:', ROBOFLOW_API_KEY[:4] + '****')

In [ ]:
# ── Cell 4 ── Download dataset ──────────────────────────────────────────────
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
DATASET_PATH = '/content/datasets/waste'

DATASETS = [
    ('material-identification',               'garbage-classification-3', 2, '10k images – PAPER/PLASTIC/GLASS/METAL/CARDBOARD [PRIMARY]'),
    ('trash-dataset-for-oriented-bounded-box', 'trash-detection',         1, '2.8k real street trash'),
    ('trash-detection-dnoed',                 '5-types-of-trash-yg5ye',   1, 'bottles, cans, bags'),
]

dataset = None
for ws, proj, ver, desc in DATASETS:
    try:
        print(f'\n📥 Trying: {proj} — {desc}')
        dataset = rf.workspace(ws).project(proj).version(ver).download(
            'yolov8', location=DATASET_PATH, overwrite=True)
        print(f'✅ Downloaded: {dataset.location}'); break
    except Exception as e:
        print(f'   ⚠️ Failed ({e}) — trying next...')

if dataset is None:
    raise RuntimeError('❌ All datasets failed. Check your API key.')

In [ ]:
# ── Cell 5 ── Fix paths + convert segmentation→bbox ────────────────────────
import yaml, glob
from pathlib import Path

yaml_path = f'{DATASET_PATH}/data.yaml'
with open(yaml_path) as f: cfg = yaml.safe_load(f)
print(f'Classes ({cfg["nc"]}):', cfg['names'])
for s in ['train','valid','val','test']:
    p = Path(DATASET_PATH)/s/'images'
    if p.exists(): print(f'  {s}: {len(list(p.glob("*")))} images')

cfg['path']  = DATASET_PATH
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images' if (Path(DATASET_PATH)/'valid/images').exists() else 'val/images'
cfg['test']  = 'test/images'  if (Path(DATASET_PATH)/'test').exists() else cfg['val']
with open(yaml_path,'w') as f: yaml.dump(cfg,f,default_flow_style=False,sort_keys=False)
print('\n✅ Paths fixed')

converted = 0
for lbl in glob.glob(f'{DATASET_PATH}/*/labels/*.txt'):
    lines = open(lbl).readlines(); nl,chg = [],False
    for line in lines:
        p = line.strip().split()
        if len(p) > 5:
            cls=p[0]; c=list(map(float,p[1:]))
            xs,ys=c[0::2],c[1::2]
            xc=(min(xs)+max(xs))/2; yc=(min(ys)+max(ys))/2
            w=max(xs)-min(xs); h=max(ys)-min(ys)
            nl.append(f'{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n'); chg=True
        else: nl.append(line)
    if chg: open(lbl,'w').writelines(nl); converted+=1
print(f'✅ Converted {converted} seg→bbox files' if converted else '✅ Labels OK (bbox format)')

In [ ]:
# ── Cell 6 ── 🚀 TRAIN (70 epochs — ~40 min — writes live to Drive) ─────────
#
# Why 70 epochs?
#   Run 1: model early-stopped at epoch 67 (fully converged)
#   Run 2: mAP@50=0.567 at epoch 113, barely better than epoch 80
#   → 70 epochs gives identical accuracy to 120, saves 25 min of GPU time
#
# project=DRIVE_RUNS → every checkpoint writes to Google Drive LIVE
# save_period=10     → Drive has weights at epoch 10, 20, 30... in case of crash

from ultralytics import YOLO
import os

model = YOLO('yolov8s.pt')

results = model.train(
    data=f'{DATASET_PATH}/data.yaml',
    epochs=70,              # ← reduced from 120 (model converges by 67)
    imgsz=640,
    batch=32,
    name='waste_v2',
    project=DRIVE_RUNS,     # ← DIRECTLY to Google Drive (live!)

    # Augmentation
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.4,
    degrees=15.0, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.05,
    mosaic=1.0,             # ★ simulates cluttered trash piles
    mixup=0.1, copy_paste=0.1,

    # Optimizer
    optimizer='AdamW', lr0=0.001, lrf=0.01, warmup_epochs=5,

    # Regularization
    dropout=0.1, weight_decay=0.0005,

    # Control
    patience=20,            # stop early if no improvement for 20 epochs
    save_period=10,         # checkpoint to Drive at epoch 10,20,30,40,50,60,70
    plots=True,
    verbose=True,
)

print(f'\n✅ Training complete!')
print(f'   best.pt → {DRIVE_RUNS}/waste_v2/weights/best.pt')

In [ ]:
# ── Cell 7 ── Results ───────────────────────────────────────────────────────
metrics = model.val()
print('\n' + '='*50 + '\n📊 Final Results\n' + '='*50)
print(f'  mAP@50:    {metrics.box.map50:.3f}  (your previous best: 0.567)')
print(f'  mAP@50-95: {metrics.box.map:.3f}')
print(f'  Precision: {metrics.box.mp:.3f}')
print(f'  Recall:    {metrics.box.mr:.3f}')
print('='*50)
if   metrics.box.map50 >= 0.55: print('\n🎉 Good model — deploy it!')
elif metrics.box.map50 >= 0.40: print('\n✅ Decent — usable in the app')
else:                            print('\n⚠️  Low accuracy')

In [ ]:
# ── Cell 7b ── Reload from Drive (if Cell 7 gives NameError) ───────────────
# Run ONLY if Cell 7 fails with 'model not defined'
from ultralytics import YOLO
import glob

found = glob.glob('/content/drive/MyDrive/waste_model/runs/waste_v2/weights/best.pt')
if not found:
    found = glob.glob('/content/drive/MyDrive/waste_model/runs/**/best.pt', recursive=True)

if found:
    model = YOLO(found[0])
    DATASET_PATH = '/content/datasets/waste'
    print(f'✅ Loaded: {found[0]}')
    print(f'Classes: {model.names}')
else:
    print('❌ No weights in Drive.')

In [ ]:
# ── Cell 8 ── Visual predictions ───────────────────────────────────────────
import glob, random
import matplotlib.pyplot as plt

test_imgs = (glob.glob(f'{DATASET_PATH}/test/images/*') or
             glob.glob(f'{DATASET_PATH}/valid/images/*') or
             glob.glob(f'{DATASET_PATH}/val/images/*'))

fig, axes = plt.subplots(2,3,figsize=(16,10))
fig.patch.set_facecolor('#0f0f1a')
for ax,p in zip(axes.flat, random.sample(test_imgs,min(6,len(test_imgs)))):
    r = model(p, conf=0.25, verbose=False)[0]
    ax.imshow(r.plot()[...,::-1]); ax.axis('off')
    ax.set_title(f'{len(r.boxes)} detections', fontsize=9, color='lime')
plt.suptitle('Model Predictions — Test Set', fontsize=14, color='white')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 9 ── Class names (for app config) ──────────────────────────────────
import yaml
with open(f'{DATASET_PATH}/data.yaml') as f: cfg = yaml.safe_load(f)
print('Classes in your trained model:')
for i,n in enumerate(cfg['names']): print(f'  {i}: {n}')

In [ ]:
# ── Cell 10 ── ⬇️ Download best.pt ──────────────────────────────────────────
#
# NOTE: best.pt is ALREADY in your Google Drive at:
#   My Drive / waste_model / runs / waste_v2 / weights / best.pt
# You can download it from Drive any time — even days later.
#
# This cell also downloads it directly to your PC right now.
#
# After downloading:
#   1. Rename to best.pt
#   2. Replace: waste-detection-main/weights/best.pt
#   3. Restart: python run_app.py

import shutil, os
from google.colab import files

candidates = [
    f'{DRIVE_RUNS}/waste_v2/weights/best.pt',
    '/content/runs/waste_v2/weights/best.pt',
]
src = next((p for p in candidates if os.path.exists(p)), None)

if src:
    dst = '/content/waste_best_FINAL.pt'
    shutil.copy(src, dst)
    print(f'📦 Downloading from: {src}')
    print('📂 Place at: waste-detection-main/weights/best.pt')
    files.download(dst)
else:
    print('❌ Not found locally — check Google Drive directly:')
    print('   My Drive / waste_model / runs / waste_v2 / weights / best.pt')